# Baseline model: predicting `real_price`

Predicts `search_summary.real_price` from the other features in `data/synthetic_data.csv` using two baselines: Linear Regression and a Decision Tree Regressor.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)

## Load data

In [ ]:
df = pd.read_csv("data/synthetic_data.csv")
print(df.shape)
df.head()

## Define target and features

Target: `search_summary.real_price`.

Dropped columns:
- `request_id` — unique identifier, not predictive.
- `search_summary.price_savings_vs_requested` — computed directly as `requested_unit_price - real_price`, so it leaks the target.

In [ ]:
target = "search_summary.real_price"
drop_cols = ["request_id", "search_summary.price_savings_vs_requested"]

X = df.drop(columns=drop_cols + [target])
y = df[target]

categorical_cols = X.select_dtypes(include="object").columns.tolist()
numeric_cols = X.select_dtypes(exclude="object").columns.tolist()

print("Categorical:", categorical_cols)
print("Numeric:", numeric_cols)

## Train/test split and preprocessing pipeline

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## Train baselines: Linear Regression and Decision Tree

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=5, random_state=42),
}

results = {}
for name, model in models.items():
    pipeline = Pipeline(steps=[("preprocess", preprocessor), ("model", model)])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)

    results[name] = {
        "pipeline": pipeline,
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": np.sqrt(mean_squared_error(y_test, preds)),
        "R2": r2_score(y_test, preds),
    }

results_df = pd.DataFrame({name: {k: v for k, v in r.items() if k != "pipeline"} for name, r in results.items()}).T
results_df

## Naive baseline for comparison

Predicting the training mean for every test row — the trained models should beat this.

In [ ]:
naive_preds = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)

print(f"Naive (mean) MAE:  {mean_absolute_error(y_test, naive_preds):,.2f}")
print(f"Naive (mean) RMSE: {np.sqrt(mean_squared_error(y_test, naive_preds)):,.2f}")

## Predicted vs actual (best model)

In [ ]:
import matplotlib.pyplot as plt

best_name = results_df["R2"].astype(float).idxmax()
best_pipeline = results[best_name]["pipeline"]
best_preds = best_pipeline.predict(X_test)

plt.figure(figsize=(6, 6))
plt.scatter(y_test, best_preds, alpha=0.5)
lims = [min(y_test.min(), best_preds.min()), max(y_test.max(), best_preds.max())]
plt.plot(lims, lims, "r--", label="Perfect prediction")
plt.xlabel("Actual real_price")
plt.ylabel("Predicted real_price")
plt.title(f"Predicted vs Actual — {best_name}")
plt.legend()
plt.tight_layout()
plt.show()